# Axial v3 Iteration A - raw_0 audit

Auditoria estructural train/validation y auditoria probabilistica validation-only. No interpreta `raw_0` como anatomia.

In [ ]:
import importlib.util
import subprocess
import sys

required = {"numpy": "numpy", "pandas": "pandas", "PIL": "Pillow", "matplotlib": "matplotlib", "torch": "torch"}
missing = [pip_package for module_name, pip_package in required.items() if importlib.util.find_spec(module_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

In [ ]:
from pathlib import Path
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
USE_DRIVE = os.getenv("PFI_USE_GOOGLE_DRIVE", "1" if IN_COLAB else "0") == "1"
if USE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("PFI_USE_GOOGLE_DRIVE=1 requiere runtime Colab")
    from google.colab import drive
    drive.mount("/content/drive")
    my_drive = Path("/content/drive/MyDrive")
    pfi_root = my_drive / "PFI_MVP"
    if not my_drive.exists() or not pfi_root.exists():
        raise FileNotFoundError("No se encontro /content/drive/MyDrive/PFI_MVP")

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_ROOT = Path(os.getenv("PFI_REPO_ROOT", "/content/PFI_MVPTest_Enzo_AImodule" if IN_COLAB else ".")).resolve()
REPO_REF = os.getenv("PFI_REPO_REF", "").strip()
REPO_URL = os.getenv("PFI_REPO_URL", "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git")
if not (REPO_ROOT / ".git").exists():
    if not REPO_REF:
        raise RuntimeError("PFI_REPO_REF debe definirse para clonar en un runtime nuevo")
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_ROOT)])
if REPO_REF:
    subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--all", "--tags"])
    subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", REPO_REF])
effective_sha = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
os.environ["PFI_REPO_ROOT"] = str(REPO_ROOT)
print({"repoRoot": str(REPO_ROOT), "effectiveSha": effective_sha})
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

In [ ]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path(os.environ["PFI_REPO_ROOT"]).resolve()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

In [ ]:
from lumbar_mri.axial_v3.guards import require_train_val_only
from lumbar_mri.axial_v3.iteration_a import AxialV3AuditConfig, run_iteration_a

ALLOWED_SPLITS = require_train_val_only(["train", "val"], context="notebook_51")
CFG = AxialV3AuditConfig()

## Protocol

The runner validates storage, reads the curated manifest, filters development records to train/validation, maps raw labels explicitly to class indices, writes structural audit tables, and optionally evaluates the v2 checkpoint on validation only.

In [ ]:
RUN_AUDIT = os.getenv("PFI_RUN_AXIAL_V3_AUDIT", "0") == "1"

if RUN_AUDIT:
    result = run_iteration_a(CFG)
    print(json.dumps(result, indent=2, default=str))
else:
    print("Iteration A preparada. Definir PFI_RUN_AXIAL_V3_AUDIT=1 para ejecutar.")